<a href="https://colab.research.google.com/github/vichu2k7-crypto/CyberSecurity-Lab/blob/main/U3e10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import hashlib
sample_file_content = (
    b"MZ\x90\x00...fake_pe_header..."
    b"CreateRemoteThread..."
    b"WriteProcessMemory..."
)

suspicious_api_keywords = [
    b"CreateRemoteThread",
    b"WriteProcessMemory",
    b"RegSetValue",
    b"InternetOpenUrl"
]


def static_analysis(file_bytes):
    file_hash = hashlib.sha256(file_bytes).hexdigest()

    found_apis = [
        api.decode()
        for api in suspicious_api_keywords
        if api in file_bytes
    ]

    return {
        "sha256": file_hash,
        "suspicious_apis_found": found_apis,
        "static_risk_score": len(found_apis)
    }


def dynamic_analysis(sandbox_log):
    indicators = 0
    reasons = []

    if sandbox_log.get("registry_writes", 0) > 0:
        indicators += 1
        reasons.append("Modified registry")

    if sandbox_log.get("network_connections", 0) > 0:
        indicators += 1
        reasons.append("Made outbound network connection")

    if sandbox_log.get("new_processes_spawned", 0) > 1:
        indicators += 1
        reasons.append("Spawned multiple child processes")

    return {
        "dynamic_risk_score": indicators,
        "reasons": reasons
    }


def classify(static_result, dynamic_result, threshold=2):
    total = (
        static_result["static_risk_score"]
        + dynamic_result["dynamic_risk_score"]
    )

    return "malicious" if total >= threshold else "benign"


# Suspicious sample
static_result = static_analysis(sample_file_content)

sandbox_log = {
    "registry_writes": 3,
    "network_connections": 1,
    "new_processes_spawned": 2
}

dynamic_result = dynamic_analysis(sandbox_log)

verdict = classify(static_result, dynamic_result)


# Benign sample
benign_static = static_analysis(
    b"just a normal text file with no suspicious content"
)

benign_dynamic = dynamic_analysis({
    "registry_writes": 0,
    "network_connections": 0,
    "new_processes_spawned": 0
})

benign_verdict = classify(benign_static, benign_dynamic)


print("Static analysis:", static_result)
print("Dynamic analysis:", dynamic_result)
print("Verdict:", verdict)
print("Benign file verdict:", benign_verdict)

Static analysis: {'sha256': '25329da60625b452686ec727bb3d0f706b638afbd6fcfda5720437ab56030222', 'suspicious_apis_found': ['CreateRemoteThread', 'WriteProcessMemory'], 'static_risk_score': 2}
Dynamic analysis: {'dynamic_risk_score': 3, 'reasons': ['Modified registry', 'Made outbound network connection', 'Spawned multiple child processes']}
Verdict: malicious
Benign file verdict: benign


In [2]:
def test_experiment10():
    assert len(static_result["sha256"]) == 64
    assert "CreateRemoteThread" in static_result["suspicious_apis_found"]
    assert static_result["static_risk_score"] >= 2
    assert dynamic_result["dynamic_risk_score"] == 3
    assert verdict == "malicious"
    assert benign_verdict == "benign"

    print("Experiment 10: All test cases passed.")


test_experiment10()

Experiment 10: All test cases passed.
